In [1]:
import pandas as pd

# 1. Read the original .xls file
# Note: Reading .xls files may require the xlrd library: pip install xlrd
file_name = '江苏境外旅客结构+消费.xls'
df = pd.read_excel(file_name)

# 2. Locate the 'Item' column and rename it to 'Country', 
# while removing the unnecessary Chinese column '项 目'
if 'Item' in df.columns:
    df_cleaned = df.rename(columns={'Item': 'Country'})
    if '项 目' in df_cleaned.columns:
        df_cleaned = df_cleaned.drop(columns=['项 目'])
else:
    # If column names are not read correctly, operate based on index
    # Assuming the first column is Chinese and the second column is English
    df_cleaned = df.iloc[:, 1:].copy() 
    df_cleaned.columns = ['Country'] + list(df.columns[2:])

# 3. Clean country names (remove leading/trailing spaces, e.g., " Japan" -> "Japan")
df_cleaned['Country'] = df_cleaned['Country'].astype(str).str.strip()

# 4. Convert "Wide Format" to "Long Format" (Melt)
# Keep 'Country' fixed, and turn all year columns into rows
df_long = df_cleaned.melt(
    id_vars=['Country'], 
    var_name='Year', 
    value_name='Visitors'
)

# 5. Data Cleaning
# Convert Year to integer (avoids formats like 2000.0)
df_long['Year'] = pd.to_numeric(df_long['Year'], errors='coerce').fillna(0).astype(int)
# Convert Visitors to integer
df_long['Visitors'] = pd.to_numeric(df_long['Visitors'], errors='coerce').fillna(0).astype(int)

# 6. Sorting (Ascending by Year and Country for better organization)
df_long = df_long.sort_values(by=['Year', 'Country']).reset_index(drop=True)

# 7. Output to CSV file
# Use 'utf-8-sig' encoding to ensure compatibility with Excel without garbled text
df_long.to_csv('cleaned_tourism_en.csv', index=False, encoding='utf-8-sig')

print("✅ Conversion successful! The generated CSV file has been saved.")

✅ Conversion successful! The generated CSV file has been saved.


In [12]:
import pandas as pd

# 1. Read the original Excel file
file_name = '江苏国内旅游人数+消费.xls'
df = pd.read_excel(file_name)

# 2. Set exact column names
df.columns.values[0] = "Indicator"
df.columns.values[1] = "Unnamed: 1"

# 3. Process Year headers (e.g., 2000.0 -> "2000")
new_columns = []
for col in df.columns:
    try:
        val = float(col)
        if val >= 2000:
            new_columns.append(str(int(val)))
        else:
            new_columns.append(col)
    except:
        new_columns.append(col)
df.columns = new_columns

# 4. Define the translation dictionary
# This covers indicators and all cities for both columns
translation_dict = {
    '国内旅游接待人数(万人次)': 'Domestic Tourists (10k)',
    '国内旅游收入(亿元)': 'Domestic Tourism Revenue (Billion CNY)',
    '南 京 市': 'Nanjing',
    '无 锡 市': 'Wuxi',
    '徐 州 市': 'Xuzhou',
    '常 州 市': 'Changzhou',
    '苏 州 市': 'Suzhou',
    '南 通 市': 'Nantong',
    '连 云 港 市': 'Lianyungang',
    '淮 安 市': 'Huai\'an',
    '盐 城 市': 'Yancheng',
    '扬 州 市': 'Yangzhou',
    '镇 江 市': 'Zhenjiang',
    '泰 州 市': 'Taizhou',
    '宿 迁 市': 'Suqian',
    '南京': 'Nanjing', '无锡': 'Wuxi', '徐州': 'Xuzhou', '常州': 'Changzhou',
    '苏州': 'Suzhou', '南通': 'Nantong', '连云港': 'Lianyungang', "淮安": "Huai'an",
    '盐城': 'Yancheng', '扬州': 'Yangzhou', '镇江': 'Zhenjiang', '泰州': 'Taizhou', '宿迁': 'Suqian'
}

# 5. Define a robust translation function
def translate_value(val):
    if pd.isna(val) or str(val).strip() == '':
        return val
    # Remove all spaces (regular and non-breaking \xa0)
    clean_val = str(val).replace(' ', '').replace('\xa0', '').strip()
    
    if clean_val in translation_dict:
        return translation_dict[clean_val]
    return val

# 6. Apply translation to BOTH the first and second columns
df['Indicator'] = df['Indicator'].apply(translate_value)
df['Unnamed: 1'] = df['Unnamed: 1'].apply(translate_value)

# 7. Final Save
# Use utf-8-sig to match the encoding of your target file
df.to_csv('JS_domestic_en.csv', index=False, encoding='utf-8-sig')

print("✅ Conversion complete! The file is now an exact match.")

✅ Conversion complete! The file is now an exact match.


In [14]:
import pandas as pd

# 1. Load the original .xls file
# Note: Ensure 'xlrd' is installed (pip install xlrd)
file_name = '江苏境外旅客总人数与总消费.xlsx'
df = pd.read_excel(file_name)

# 2. Extract Year columns (skipping the first two descriptive columns)
# The year headers start from index 2
years = df.columns[2:]

# 3. Extract the data rows
# Row 0 is Revenue, Row 1 is Tourists
revenue_data = df.iloc[0, 2:].values
tourist_data = df.iloc[1, 2:].values

# 4. Create the new structured DataFrame
# We align the data to the columns: Year, Revenue (10k USD), Tourists (Person)
df_final = pd.DataFrame({
    'Year': years,
    'Revenue (10k USD)': revenue_data,
    'Tourists (Person)': tourist_data
})

# 5. Data Cleaning
# Convert Year to integer (e.g., 2000.0 -> 2000)
df_final['Year'] = pd.to_numeric(df_final['Year'], errors='coerce').astype(int)

# Ensure values are numeric (integers)
df_final['Revenue (10k USD)'] = pd.to_numeric(df_final['Revenue (10k USD)'], errors='coerce').astype(int)
df_final['Tourists (Person)'] = pd.to_numeric(df_final['Tourists (Person)'], errors='coerce').astype(int)

# 6. Save to CSV
# Using index=False to match your target file exactly
df_final.to_csv('JS_foreign_en.csv', index=False, encoding='utf-8-sig')

print("✅ Success! 'JS_foreign_en.csv' has been generated with transposed data.")

✅ Success! 'JS_foreign_en.csv' has been generated with transposed data.


In [15]:
import pandas as pd

# 1. Load the original .xls file
file_name = '江苏国内旅游人数+消费.xls'
df = pd.read_excel(file_name)

# 2. Define translation mapping for cities
city_map = {
    '南 京 市': 'Nanjing',
    '无 锡 市': 'Wuxi',
    '徐 州 市': 'Xuzhou',
    '常 州 市': 'Changzhou',
    '苏 州 市': 'Suzhou',
    '南 通 市': 'Nantong',
    '连 云 港 市': 'Lianyungang',
    '淮 安 市': 'Huai\'an',
    '盐 城 市': 'Yancheng',
    '扬 州 市': 'Yangzhou',
    '镇 江 市': 'Zhenjiang',
    '泰 州 市': 'Taizhou',
    '宿 迁 市': 'Suqian'
}

# 3. Handle Year headers (convert float 2000.0 to string "2000")
new_columns = []
for col in df.columns:
    try:
        val = float(col)
        if val >= 2000:
            new_columns.append(str(int(val)))
        else:
            new_columns.append(col)
    except:
        new_columns.append(col)
df.columns = new_columns

# 4. Prepare data for the map CSV
final_rows = []

# We need to track which metric section we are in
# The first half of the file is Visitors, the second half is Revenue
current_type = "Visitors"

for index, row in df.iterrows():
    raw_name = str(row.iloc[0]).replace(' ', '').strip()
    
    # Check if we switched to the Revenue section
    if '收入' in raw_name and '国内旅游' in raw_name:
        current_type = "Revenue"
        continue
    if '人数' in raw_name and '国内旅游' in raw_name:
        current_type = "Visitors"
        continue
        
    # Map the city name
    clean_city_key = str(row.iloc[0]).strip()
    if clean_city_key in city_map:
        new_row = {
            'city': city_map[clean_city_key],
            'type': current_type
        }
        # Add year data to the row
        for col in df.columns:
            if col.isdigit():
                new_row[col] = row[col]
        final_rows.append(new_row)

# 5. Create final DataFrame
df_final = pd.DataFrame(final_rows)

# 6. Save to CSV
# Using utf-8-sig for Excel compatibility
df_final.to_csv('js_map_data_en_real.csv', index=False, encoding='utf-8-sig')

print(f"✅ Success! 'js_map_data_en_real.csv' generated with {len(df_final)} city records.")

✅ Success! 'js_map_data_en_real.csv' generated with 24 city records.


In [17]:
import pandas as pd

# 1. Load the original Excel file
file_name = '江苏省景点位置 - 副本.xlsx'
df = pd.read_excel(file_name)

# 2. Define city translation mapping
city_map = {
    '南京': 'Nanjing', '无锡': 'Wuxi', '徐州': 'Xuzhou',
    '常州': 'Changzhou', '苏州': 'Suzhou', '南通': 'Nantong',
    '连云港': 'Lianyungang', '淮安': 'Huai\'an', '盐城': 'Yancheng',
    '扬州': 'Yangzhou', '镇江': 'Zhenjiang', '泰州': 'Taizhou', '宿迁': 'Suqian'
}

# 3. Create the target structure
df_final = pd.DataFrame()

# Map basic text columns
df_final['Name'] = df['名字']
df_final['Address'] = df['地址']
df_final['City'] = df['所属城市'].map(city_map)

# 4. Split WGS84 Coordinates into lng and lat
# Original format: "118.738472, 31.907540"
def split_coords(coord_str, index):
    if pd.isna(coord_str) or ',' not in str(coord_str):
        return ""
    try:
        parts = str(coord_str).split(',')
        return parts[index].strip()
    except:
        return ""

# Apply splitting (Index 0 is lng, Index 1 is lat)
df_final['lng'] = df['WGS84经纬度'].apply(lambda x: split_coords(x, 0))
df_final['lat'] = df['WGS84经纬度'].apply(lambda x: split_coords(x, 1))

# Map other columns
df_final['Phone'] = df['电话']
df_final['Website'] = df['官网']
df_final['Introduction'] = df['介绍']
df_final['Open_Time'] = df['开放时间']
df_final['Rating'] = df['评分']
df_final['Suggested_Duration'] = df['建议游玩时间']

# Duplicate coordinates for lng.1 and lat.1 as per your target sample
df_final['lng.1'] = df_final['lng']
df_final['lat.1'] = df_final['lat']

# 5. Data Cleaning
# Replace any NaN with empty strings
df_final = df_final.fillna("")

# 6. Save to CSV
df_final.to_csv('JS_scenic_en.csv', index=False, encoding='utf-8-sig')

print("✅ Success! 'JS_scenic_en.csv' has been generated with split coordinates.")

✅ Success! 'JS_scenic_en.csv' has been generated with split coordinates.
